# نسخه۸: از پایلوت تا TEST مقاله
این نوت‌بوک از paper_v8.py استفاده می‌کند. تنظیم مطالعه ذخیره می‌شود؛ پس از scan، دستور test همان تنظیمات را بازیابی می‌کند. سلول‌های آموزش با False تحویل داده شده‌اند تا اجرای همهٔ سلول‌ها ناخواسته مطالعهٔ سنگین آغاز نکند. راهنمای کامل: docs/V8_RUNBOOK_FA.md. معادل کامل راهنمای نسخه۷، همراه PowerShell و سه سطح اجرا: docs/V8_RELEASE_RUNBOOK_FA.md. معنی مراحل: docs/V8_COMMAND_REFERENCE_FA.md. شاخهٔ انتشار v8 و شاخهٔ توسعه v8_development است.


In [ ]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd

PROJECT = Path.cwd()
if PROJECT.name == 'notebooks':
    PROJECT = PROJECT.parent
assert (PROJECT/'paper_v8.py').is_file(), 'Set PROJECT to the repository root.'
DATA_ROOT = PROJECT/'datasets'  # May point to the old V7 datasets directory.
DEVICE = 'cuda'  # Use 'cpu' or 'auto' if desired.
TUNED = PROJECT/'results_v8/tuned_light'

def command(*args):
    return subprocess.run([sys.executable,*map(str,args)],cwd=PROJECT,check=True)

def paper(stage, output, *extra, initial=False):
    args=['paper_v8.py','--stage',stage,'--output',str(output)]
    if initial:
        args += ['--profile','common','--budget','light','--datasets','ml-100k',
                 '--data-root',str(DATA_ROOT),'--device',DEVICE]
    return command(*args,*extra)

command('verify_v8.py')


## ۱. smoke: TEST کوتاه برای سلامت فنی
شکل‌ها watermark دارند؛ این عددها نتیجهٔ دقت مقاله نیستند.


In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    paper('smoke',PROJECT/'results_v8/smoke',initial=True)


## ۲. پایلوت ثابت: TRAIN/VAL و TEST دارد
configهای پیش‌فرض را می‌سنجد؛ جست‌وجوی config انجام نمی‌دهد. خروجی زیر main است.


In [ ]:
RUN_PILOT = False
if RUN_PILOT:
    paper('pilot',PROJECT/'results_v8/pilot',initial=True)


## ۳. طرح مطالعهٔ تنظیم‌شده
این مرحله آموزش نمی‌دهد؛ recipe را ذخیره می‌کند. مراحل بعدی تنها output را لازم دارند. برای تغییر علمی مطالعه، پوشهٔ تازه بسازید.


In [ ]:
RUN_PLAN = False
if RUN_PLAN:
    paper('plan',TUNED,initial=True)


## ۴. جست‌وجو فقط روی VAL
اگر plan را نگرفته‌اید، initial=True را نگه دارید؛ در ادامهٔ مطالعه نیز تنظیمات صریح یکسان معتبرند. هنوز TEST تولید نمی‌شود.


In [ ]:
RUN_SCAN = False
if RUN_SCAN:
    paper('scan',TUNED,initial=True)


## ۵. این سلول TEST واقعی را می‌گیرد
این همان final برندگان scan است. هر هشت روش شما و baselineهای مجاز مطالعه حضور دارند؛ فهرست روش‌ها را برای دورزدن خطا حذف نمی‌کنیم.


In [ ]:
RUN_TEST = False
if RUN_TEST:
    paper('test',TUNED)


In [ ]:
base=TUNED/'ml-100k'
f=base/'final_locked_test_summary.csv'
if f.is_file():
    table=pd.read_csv(f)
    display(table[[c for c in ['method','test_RMSE_mean','test_RMSE_std','test_MAE_mean','n_outer_splits'] if c in table]])
else:
    print('TEST has not completed:',f)
image=base/'publication/figures/selected_test_rmse.png'
if image.is_file():
    from IPython.display import Image,display
    display(Image(filename=str(image)))


## ۶. گزارش و شکل‌ها بدون آموزش مجدد
اگر TRAIN را با no-plots گرفته‌اید، plots اینجا آن را فعال می‌کند. از دادهٔ VAL، TEST ساختگی تولید نمی‌شود.


In [ ]:
RUN_REPORT = False
if RUN_REPORT:
    paper('report',TUNED,'--plots')
RUN_STATUS = False
if RUN_STATUS:
    paper('status',TUNED)


## ۷. حساسیت ضریب transport متد۶ فقط با VAL
این مرحله تنظیم برنده را خودکار عوض نمی‌کند؛ آزمایش تک‌عاملی است. برای همهٔ ضرایب، دو گزینهٔ فیلتر را حذف کنید.


In [ ]:
RUN_LOSS_STUDY = False
if RUN_LOSS_STUDY:
    paper('lossstudy',TUNED,'--loss-methods','Method6','--loss-parameters','transport_moment_weight')


## ۸. حذف مؤلفه، همگرایی و probability
ablation و sparsity با تنظیمات منجمد و TEST توصیفی‌اند. convergence فقط TRAIN/VAL است. calibration مطالعهٔ جدا با انتخاب VAL NLL برای متد۷ است.


In [ ]:
RUN_SECONDARY = False
if RUN_SECONDARY:
    for stage in ['ablation','ablation-fixed','convergence','sparsity','calibration']:
        paper(stage,TUNED)
    paper('audit',TUNED)


## ۹. مطالعهٔ میانی مقاله، در خروجی تازه
این مطالعه پروتکل aemc-paper و متد۷ point-only دارد. برای احتمال، common جدا لازم است. تعداد split/seed نمونه‌های مستقل تضمین‌شده نیست. ابتدا فقط plan؛ all ممکن است بسیار پرهزینه باشد.


In [ ]:
PAPER = PROJECT/'results_v8/paper_medium'
RUN_PAPER_PLAN = False
if RUN_PAPER_PLAN:
    command('paper_v8.py','--stage','plan','--profile','paper',
            '--datasets','ml-100k','--data-root',DATA_ROOT,'--output',PAPER,
            '--device',DEVICE,'--split-seeds','4101,4102,4103',
            '--model-seeds','11,22','--k','10,20')
RUN_PAPER_SUITE = False
if RUN_PAPER_SUITE:
    paper('all',PAPER,'--include-temporal')


## ۱۰. معادل curves نسخه۷، بدون آموزش
این سلول history موجود را رسم می‌کند؛ برای RMSE کامل TRAIN که قبلاً ذخیره نشده، از convergence استفاده کنید.


In [ ]:
RUN_CURVES = False
if RUN_CURVES:
    paper('curves',TUNED,'--plots')


## ۱۱. شمارش برنامه و تخمین زمان با پایلوت دستگاه خودتان
این تخمین آموزش نمی‌دهد. مرجع پیش‌فرض از GPU دیگری است؛ برای تخمین مرتبط با دستگاه خودتان، calibration را روی CSV پایلوت همان دستگاه بگذارید.


In [ ]:
RUN_ESTIMATE = False
if RUN_ESTIMATE:
    estimate_args=['estimate_v8.py','--profile','common',
                   '--search-jobs-per-method','6','--max-epochs','120',
                   '--steps-per-epoch','2','--baseline-max-epochs','200']
    measured=PROJECT/'results_v8/pilot/ml-100k/main/candidate_validation_runs.csv'
    if measured.is_file():
        estimate_args += ['--calibration',str(measured)]
    command(*estimate_args)
